# Jeveration — source-faithful LiDeRe + Probe3D (95 images)

The exact published **LiDeRe** implicit spatial-attention/upscaling source (MIT) and exact **Probe3D** depth-bin / depth / normal loss source (MIT) are included in Jeveration. Three independently optimized readouts, unchanged 95 train / 20 validation Hypersim cache, and **1,200 optimizer updates each**. The previous approximate frozen-probe experiment remains untouched.

**Note:** This is a faithful port of the published readout and loss implementations to *Jeveration's cached frozen features*, not a reproduction of their paper's backbone or performance. Expect higher GPU costs than the old random-projection probe.

**Installation safety:** Cell 1 force-reinstalls the pinned revision even if the package version number is unchanged. Cell 3 independently detects and repairs the old CLI parser before preflight or training.

In [ ]:
# @title 1. Mount Drive and FORCE install verified LiDeRe training code
from google.colab import drive
drive.mount("/content/drive")
import sys, subprocess, importlib, argparse, torch
from importlib.metadata import version, PackageNotFoundError

PINNED_LIDERE_COMMIT = "7d918dba6debb7dfc8d0e5dbafd6d634549fa87f"
SOURCE_URL = f"git+https://github.com/Apache0ne/Jeveration.git@{PINNED_LIDERE_COMMIT}"

# First-time installations need the package's dependencies. For existing
# Colab sessions, DO NOT reinstall all large Torch dependencies needlessly.
try:
    version("apache0ne-jeveration")
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", SOURCE_URL], check=True)

# --upgrade alone can KEEP THE OLD CODE since pyproject's version is unchanged.
# Force-replace the installed code from the verified Git commit, always.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "--force-reinstall", "--no-deps", "--no-cache-dir", SOURCE_URL],
               check=True)

importlib.invalidate_caches()
sys.modules.pop("jeveration.rgbd_source_lidere_train", None)
trainer = importlib.import_module("jeveration.rgbd_source_lidere_train")
for name in ("preflight", "fit", "report"):
    parsed = trainer.parse_args([name])
    assert isinstance(parsed, argparse.Namespace) and parsed.command == name, (
        "Installed LiDeRe code is stale: CLI argument parsing failed")
print("VERIFIED INSTALLED FILE:", trainer.__file__)
print("LiDeRe commands: preflight / fit / report — all parsed correctly")
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), "Select BF16 L4 or A100"
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# @title 2. Verify existing 95/20 cache and L4 settings
from pathlib import Path
ROOT=Path("/content/drive/MyDrive/Jeveration_GT/pilot")
OUT=ROOT/"checkpoints_source_lidere"
LOCAL=Path("/content/jeveration_gt_pilot_cache")
assert len(list((ROOT/"cache/train").glob("*.npz")))==95
assert len(list((ROOT/"cache/val").glob("*.npz")))==20
OUT.mkdir(parents=True,exist_ok=True)
STEPS=1200 # @param {type:"integer"}
GRAD_ACCUM=16 # @param {type:"integer"}
LIDERE_DIM=64 # @param {type:"integer"}
LIDERE_BASE_SIZE=64 # @param {type:"integer"}
assert STEPS==1200 and LIDERE_DIM%8==0
print("Original 95 train, 20 validation, zero data changes")
print("New checkpoint directory:",OUT)


In [ ]:
# @title 3. Verify installed code, preflight, then 1,200 independent updates
import sys, subprocess, importlib, argparse, traceback

# This guard also repairs an old in-memory module if you run cell 3 directly.
PINNED_LIDERE_COMMIT = "7d918dba6debb7dfc8d0e5dbafd6d634549fa87f"

def verified_lidere_trainer():
    name = "jeveration.rgbd_source_lidere_train"
    importlib.invalidate_caches()
    module = importlib.import_module(name)
    def cli_valid(mod):
        try:
            return all(
                isinstance(mod.parse_args([c]), argparse.Namespace)
                and mod.parse_args([c]).command == c
                for c in ("preflight", "fit", "report"))
        except (AttributeError, TypeError, SystemExit):
            return False

    if not cli_valid(module):
        print("STALE LiDeRe package detected. Force-installing corrected commit...",flush=True)
        subprocess.run([
            sys.executable, "-m", "pip", "install", "-q",
            "--force-reinstall", "--no-deps", "--no-cache-dir",
            f"git+https://github.com/Apache0ne/Jeveration.git@{PINNED_LIDERE_COMMIT}"
        ],check=True)
        sys.modules.pop(name, None)
        importlib.invalidate_caches()
        module = importlib.import_module(name)

    if not cli_valid(module):
        raise RuntimeError(f"LiDeRe installation is still stale at {module.__file__}")
    print("Verified correct CLI from:",module.__file__,flush=True)
    return module

def run_lidere_in_notebook(cli_args):
    module=verified_lidere_trainer()
    old_argv=sys.argv[:]
    sys.argv=["jeveration.rgbd_source_lidere_train"]+list(cli_args)
    try:
        # Execute the already VERIFIED module, not runpy's separate import.
        module.main()
    except BaseException:
        details=traceback.format_exc()
        print("\n====== FULL TRAINER TRACEBACK ======",flush=True)
        print(details[-16000:],flush=True)
        try:
            error_file=OUT/"last_lidere_error.txt"
            error_file.write_text(details,encoding="utf8")
            print("Traceback saved:",error_file,flush=True)
        except Exception as saving_error:
            print("Could not save traceback:",saving_error,flush=True)
        raise
    finally:
        sys.argv=old_argv

base=["--root",str(ROOT),"--save-dir",str(OUT),
      "--local-cache-dir",str(LOCAL),"--device","cuda"]
preflight=["preflight",*base,"--dim",str(LIDERE_DIM),
           "--base-size",str(LIDERE_BASE_SIZE),"--max-depth","80"]
print("Preflight on real cached data...",flush=True)
run_lidere_in_notebook(preflight)

cmd=["fit",*base,"--cache-mode","ram","--steps",str(STEPS),
     "--batch-size","1","--grad-accum",str(GRAD_ACCUM),
     "--dim",str(LIDERE_DIM),"--base-size",str(LIDERE_BASE_SIZE),
     "--max-depth","80","--lr-depth","0.00015",
     "--lr-normal","0.00015","--lr-seg","0.00015",
     "--eval-every","100","--save-every","200","--save-at","400,700,900"]
if (OUT/"latest_source.json").exists():
    cmd.append("--resume")
run_lidere_in_notebook(cmd)


In [ ]:
# @title 4. Export the same 20-image holdout report and copy to Drive
import shutil, json
from pathlib import Path
from google.colab import files
run_lidere_in_notebook([
    "report","--root",str(ROOT),"--save-dir",str(OUT),
    "--local-cache-dir",str(LOCAL),"--device","cuda",
    "--output","/content/Jeveration_Source_LiDeRe_Report"
])
report=Path("/content/Jeveration_Source_LiDeRe_Probe3D_Report.zip")
assert report.is_file()
dest=ROOT/"reports"/report.name
dest.parent.mkdir(parents=True,exist_ok=True)
shutil.copy2(report,dest)
metrics=json.loads(Path("/content/Jeveration_Source_LiDeRe_Report/metrics.json").read_text())
print(json.dumps(metrics,indent=2))
print("Report saved:",dest)
files.download(str(report))
